# 🛢️ Naija-Petro — NVIDIA Data Designer Pipeline (v2)

**Notebook 2 of 5 · Naija-Petro project** — fast, resumable, auto-saving synthetic data generation.

**What this notebook does:**
- Reads the scraped seed corpus from Google Drive (produced by notebook 1)
- Generates 20,000+ instruction–response pairs with NVIDIA Data Designer
- **Auto-saves every batch** to Google Drive (never lose progress)
- **Resumes from the last checkpoint** if Colab disconnects
- **Performance-tuned**: higher parallelism, optimised buffer sizes, faster models
- Runs LLM-as-Judge quality scoring on a sample
- Exports the final dataset in Alpaca + ShareGPT formats

### Key improvements over v1
| Area | v1 | v2 (this notebook) |
|-------|----|-----------|
| Speed | Default parallelism (4 concurrent) | 16–32 concurrent requests + larger buffers |
| Saving | Only at the very end | After every batch (1,000 records) |
| Resume | Start from scratch if interrupted | Detects existing batches and continues |
| Error handling | Crashes lose everything | Retry logic + partial results preserved |
| Model | Default nvidia-text | Configurable with tuned inference params |

---

## 0. Setup & Install

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install data-designer jsonlines pandas tqdm -q
!pip install "pyarrow>=19.0.1,<20" -q
print("✅ Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.4/560.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.1/947.1 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import json
import time
import glob
import hashlib
import traceback
import pandas as pd
import jsonlines
from pathlib import Path
from datetime import datetime
from collections import Counter

print("✅ All imports loaded.")

✅ All imports loaded.


In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted.")

Mounted at /content/drive
✅ Google Drive mounted.


In [ ]:
# ============================================================
# CONFIGURE PATHS (same structure as scraping notebook)
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/petroleum_corpus")

OUTPUT_DIR    = DRIVE_BASE
RAW_DIR       = OUTPUT_DIR / "raw"
PROCESSED_DIR = OUTPUT_DIR / "processed"
METADATA_DIR  = OUTPUT_DIR / "metadata"
AUGMENTED_DIR = OUTPUT_DIR / "augmented"

# NEW: checkpoint directories for resumable generation
CHECKPOINT_DIR = AUGMENTED_DIR / "checkpoints"
P1_CKPT_DIR    = CHECKPOINT_DIR / "pipeline1"
P2_CKPT_DIR    = CHECKPOINT_DIR / "pipeline2"

for d in [RAW_DIR, PROCESSED_DIR, METADATA_DIR, AUGMENTED_DIR,
          CHECKPOINT_DIR, P1_CKPT_DIR, P2_CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# GENERATION SETTINGS — adjust these to your needs
# ============================================================
NUM_IR_RECORDS   = 20000   # Pipeline 1: knowledge-based instruction-response
NUM_SEED_RECORDS = 20000   # Pipeline 2: seed-grounded QA
BATCH_SIZE       = 1000    # Records per batch (saved to Drive after each)
JUDGE_SAMPLE     = 500     # Records to quality-score

print(f"✅ Paths configured. Output: {OUTPUT_DIR}")
print(f"   Pipeline 1 target: {NUM_IR_RECORDS:,} records")
print(f"   Pipeline 2 target: {NUM_SEED_RECORDS:,} records")
print(f"   Batch size: {BATCH_SIZE:,} (auto-saves after each batch)")

✅ Paths configured. Output: /content/drive/MyDrive/petroleum_corpus
   Pipeline 1 target: 20,000 records
   Pipeline 2 target: 20,000 records
   Batch size: 1,000 (auto-saves after each batch)


In [ ]:
# ============================================================
# CONFIGURE API KEY
# ============================================================
from google.colab import userdata

# Try NVIDIA first (recommended — free at https://build.nvidia.com)
try:
    NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
    if NVIDIA_API_KEY:
        os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
except Exception:
    pass

# Fallback: OpenAI
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Fallback: OpenRouter
# os.environ["OPENROUTER_API_KEY"] = "..."

has_nvidia    = bool(os.environ.get('NVIDIA_API_KEY'))
has_openai    = bool(os.environ.get('OPENAI_API_KEY'))
has_openrouter = bool(os.environ.get('OPENROUTER_API_KEY'))

if has_nvidia:
    print("✅ Using NVIDIA Build API")
elif has_openai:
    print("✅ Using OpenAI API")
elif has_openrouter:
    print("✅ Using OpenRouter API")
else:
    print("⚠️  No API key found! Set NVIDIA_API_KEY in Colab secrets.")
    print("   Get a free key at: https://build.nvidia.com")

✅ Using NVIDIA Build API


## 1. Initialize Data Designer with Performance Tuning

In [ ]:
# ============================================================
# INITIALIZE DATA DESIGNER WITH TUNED SETTINGS
# ============================================================
import data_designer.config as dd
from data_designer.interface import DataDesigner

# --- Performance-tuned RunConfig ---
run_config = dd.RunConfig(
    buffer_size=2000,
    non_inference_max_parallel_workers=8,
    max_conversation_restarts=5,
    max_conversation_correction_steps=1,
    disable_early_shutdown=True,
)

# --- Initialize ---
data_designer = DataDesigner()
data_designer.set_run_config(run_config)

# --- Optional: Custom model with higher parallelism ---
# Uncomment the block below to use a faster model config.
# The default "nvidia-text" uses max_parallel_requests=4.

CUSTOM_MODEL = None  # Set to a ModelConfig to use a custom model

# Uncomment below to enable:
CUSTOM_MODEL = dd.ModelConfig(
     alias="fast-nvidia-text",
     model="nvidia/nemotron-3-nano-30b-a3b",
     provider="nvidia",
     inference_parameters=dd.ChatCompletionInferenceParams(
         max_parallel_requests=32,
         temperature=0.7,
        top_p=0.95,
        max_tokens=1024,
    ),
)

MODEL_ALIAS = CUSTOM_MODEL.alias if CUSTOM_MODEL else "nvidia-text"

print(f"✅ Data Designer initialized.")
print(f"   buffer_size={run_config.buffer_size}")
print(f"   model_alias={MODEL_ALIAS}")

✅ Data Designer initialized.
   buffer_size=2000
   model_alias=fast-nvidia-text


## 2. Consolidate Seed Data (from scraping notebook)

This reads the raw JSONL files already on Google Drive and prepares the seed corpus.
If `seed_corpus.parquet` already exists, it skips re-processing.

In [ ]:
# ============================================================
# CONSOLIDATE SEED DATA (skip if already done)
# ============================================================

seed_path = PROCESSED_DIR / "seed_corpus.parquet"

if seed_path.exists():
    print(f"✅ Seed corpus already exists: {seed_path}")
    seed_df = pd.read_parquet(seed_path)
    print(f"   {len(seed_df):,} chunks loaded.")
else:
    print("📂 Building seed corpus from raw JSONL files...")

    all_records = []
    source_counts = {}

    for f in sorted(RAW_DIR.glob("*.jsonl")):
        count = 0
        with jsonlines.open(f) as reader:
            for rec in reader:
                all_records.append(rec)
                count += 1
        source_counts[f.stem] = count
        print(f"   ✅ {f.stem}: {count:,}")

    print(f"\n📊 Total before dedup: {len(all_records):,}")

    # Deduplicate
    seen, unique, dupes = set(), [], 0
    for rec in all_records:
        text = rec.get('text', '')
        if not text:
            continue
        h = hashlib.md5(text[:500].lower().encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(rec)
        else:
            dupes += 1

    filtered = [r for r in unique if len(r.get('text', '')) >= 100]
    print(f"🗑️  Removed {dupes:,} duplicates + {len(unique)-len(filtered):,} short records")
    print(f"✅ Unique records: {len(filtered):,}")

    # Chunk
    chunked = []
    chunk_size, overlap = 600, 80

    for r in filtered:
        text = r.get('text', '')
        words = text.split()
        title = r.get('title', r.get('term', ''))
        source = r.get('source', 'unknown')
        content_type = r.get('content_type', 'unknown')

        if len(words) <= chunk_size * 1.3:
            chunked.append({
                "text": text, "title": title,
                "source": source, "content_type": content_type,
                "word_count": len(words),
            })
        else:
            start = 0
            while start < len(words):
                end = min(start + chunk_size, len(words))
                chunk_text = ' '.join(words[start:end])
                chunked.append({
                    "text": chunk_text, "title": title,
                    "source": source, "content_type": content_type,
                    "word_count": len(chunk_text.split()),
                })
                start += chunk_size - overlap
                if end >= len(words):
                    break

    seed_df = pd.DataFrame(chunked)
    seed_df.to_parquet(seed_path, index=False)

    # Also save JSONL backup
    with jsonlines.open(PROCESSED_DIR / "seed_corpus.jsonl", mode='w') as w:
        for rec in chunked:
            w.write(rec)

    # Stats
    stats = {
        "total_raw_records": len(all_records),
        "unique_records": len(filtered),
        "total_chunks": len(chunked),
        "total_words": int(seed_df['word_count'].sum()),
        "avg_chunk_words": int(seed_df['word_count'].mean()),
        "source_counts": source_counts,
    }
    with open(METADATA_DIR / "seed_statistics.json", 'w') as f:
        json.dump(stats, f, indent=2)

    print(f"\n✅ Seed corpus ready: {len(chunked):,} chunks")
    print(f"   Total words: {stats['total_words']:,}")
    print(f"   Saved to: {seed_path}")

✅ Seed corpus already exists: /content/drive/MyDrive/petroleum_corpus/processed/seed_corpus.parquet
   66,905 chunks loaded.


---
## 3. Helper Functions: Batched Generation with Checkpointing

The core improvement: instead of one massive `create()` call,
we loop in batches, saving each batch to Drive immediately.

In [ ]:
# ============================================================
# BATCHED GENERATION WITH AUTO-SAVE & RESUME
# ============================================================

def count_existing_checkpoints(ckpt_dir: Path) -> tuple:
    """
    Count existing checkpoint files and total records already generated.
    Returns (num_batches_done, total_records_done, list_of_checkpoint_files).
    """
    ckpt_files = sorted(ckpt_dir.glob("batch_*.parquet"))
    total_records = 0
    for f in ckpt_files:
        try:
            df = pd.read_parquet(f)
            total_records += len(df)
        except Exception:
            pass  # corrupted checkpoint, will be regenerated
    return len(ckpt_files), total_records, ckpt_files


def generate_batched(
    data_designer,
    config_builder,
    total_records: int,
    batch_size: int,
    ckpt_dir: Path,
    dataset_name: str,
):
    """
    Generate records in batches with checkpointing to Google Drive.

    - Detects existing checkpoints and resumes from where it left off.
    - Saves each batch as a separate Parquet + JSONL file to Drive.
    - On error, preserves all completed batches and reports progress.

    Returns the combined DataFrame of all generated records.
    """
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # --- Check for existing progress ---
    done_batches, done_records, existing_files = count_existing_checkpoints(ckpt_dir)
    remaining = total_records - done_records

    if done_records >= total_records:
        print(f"✅ Already completed! {done_records:,} records in {done_batches} batches.")
        print(f"   Loading from checkpoints...")
        return _load_all_checkpoints(ckpt_dir)

    if done_records > 0:
        print(f"🔄 RESUMING from batch {done_batches + 1}")
        print(f"   Already completed: {done_records:,} / {total_records:,} records")
        print(f"   Remaining: {remaining:,} records")
    else:
        print(f"🚀 Starting fresh: {total_records:,} records in batches of {batch_size:,}")

    # --- Generate remaining batches ---
    batch_num = done_batches + 1
    generated_so_far = done_records
    start_time = time.time()

    while generated_so_far < total_records:
        this_batch_size = min(batch_size, total_records - generated_so_far)
        batch_start = time.time()

        print(f"\n{'='*60}")
        print(f"📦 Batch {batch_num} — generating {this_batch_size:,} records...")
        print(f"   Progress: {generated_so_far:,} / {total_records:,} "
              f"({100*generated_so_far/total_records:.1f}%)")

        try:
            results = data_designer.create(
                config_builder=config_builder,
                num_records=this_batch_size,
                dataset_name=f"{dataset_name}_batch{batch_num:03d}",
            )
            df_batch = results.load_dataset()

            if df_batch is None or len(df_batch) == 0:
                print(f"   ⚠️  Batch {batch_num} returned 0 records. Retrying...")
                time.sleep(10)
                continue

            # --- Save checkpoint to Drive immediately ---
            ckpt_parquet = ckpt_dir / f"batch_{batch_num:03d}.parquet"
            ckpt_jsonl   = ckpt_dir / f"batch_{batch_num:03d}.jsonl"

            df_batch.to_parquet(ckpt_parquet, index=False)
            with jsonlines.open(ckpt_jsonl, mode='w') as w:
                for _, row in df_batch.iterrows():
                    w.write(row.to_dict())

            elapsed = time.time() - batch_start
            generated_so_far += len(df_batch)
            rate = len(df_batch) / elapsed * 60 if elapsed > 0 else 0

            print(f"   ✅ Batch {batch_num}: {len(df_batch):,} records in {elapsed:.0f}s "
                  f"({rate:.0f} rec/min)")
            print(f"   💾 Saved to Drive: {ckpt_parquet.name}")
            print(f"   📊 Total progress: {generated_so_far:,} / {total_records:,} "
                  f"({100*generated_so_far/total_records:.1f}%)")

            # ETA
            total_elapsed = time.time() - start_time
            records_in_this_session = generated_so_far - done_records
            if records_in_this_session > 0:
                eta_seconds = (total_records - generated_so_far) * (total_elapsed / records_in_this_session)
                eta_min = eta_seconds / 60
                print(f"   ⏱️  ETA: ~{eta_min:.0f} min remaining")

            batch_num += 1

        except KeyboardInterrupt:
            print(f"\n⏸️  Interrupted by user after {generated_so_far:,} records.")
            print(f"   All completed batches are saved. Re-run this cell to resume.")
            break

        except Exception as e:
            print(f"   ❌ Error in batch {batch_num}: {type(e).__name__}: {e}")
            traceback.print_exc()
            print(f"   ⏳ Waiting 30s before retry...")
            time.sleep(30)
            # Don't increment batch_num — retry the same batch
            continue

    # --- Load and combine all checkpoints ---
    total_elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✅ Generation complete! Session time: {total_elapsed/60:.1f} min")
    return _load_all_checkpoints(ckpt_dir)


def _load_all_checkpoints(ckpt_dir: Path) -> pd.DataFrame:
    """Load and combine all checkpoint Parquet files."""
    ckpt_files = sorted(ckpt_dir.glob("batch_*.parquet"))
    if not ckpt_files:
        return pd.DataFrame()

    dfs = []
    for f in ckpt_files:
        try:
            dfs.append(pd.read_parquet(f))
        except Exception as e:
            print(f"   ⚠️  Skipping corrupted checkpoint: {f.name} ({e})")

    combined = pd.concat(dfs, ignore_index=True)
    print(f"   📦 Loaded {len(combined):,} records from {len(dfs)} checkpoint files.")
    return combined


print("✅ Batched generation helpers ready.")

✅ Batched generation helpers ready.


---
## 4. Pipeline 1: Knowledge-Based Instruction-Response Generation

Generates diverse Q&A pairs from sampler-driven categories — no seed data needed.
Each batch is saved to Drive immediately after completion.

In [ ]:
# ============================================================
# CONFIGURE PIPELINE 1
# ============================================================

config_ir = dd.DataDesignerConfigBuilder()
# Register custom model if defined
if CUSTOM_MODEL:
    config_ir.add_model_config(CUSTOM_MODEL)  # use config_seed / config_judge for the other pipelines
# ─── Sampler: Instruction category ───
config_ir.add_column(
    dd.SamplerColumnConfig(
        name="instruction_category",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "concept_explanation",
                "calculation_problem",
                "procedure_description",
                "troubleshooting",
                "comparison_analysis",
                "design_recommendation",
                "data_interpretation",
                "best_practices",
                "safety_compliance",
                "case_study_analysis",
            ],
        ),
    )
)

# ─── Sampler: Complexity level ───
config_ir.add_column(
    dd.SamplerColumnConfig(
        name="complexity_level",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["beginner", "intermediate", "advanced", "expert"],
        ),
    )
)

# ─── Sampler: Petroleum subdomain ───
config_ir.add_column(
    dd.SamplerColumnConfig(
        name="subdomain",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "reservoir_engineering",
                "drilling_engineering",
                "production_engineering",
                "completion_stimulation",
                "formation_evaluation",
                "flow_assurance",
                "enhanced_oil_recovery",
                "offshore_subsea",
                "process_facilities",
                "petroleum_geoscience",
                "well_integrity_safety",
                "digital_oilfield_AI",
            ],
        ),
    )
)

# ─── LLM: Generate instruction ───
config_ir.add_column(
    dd.LLMTextColumnConfig(
        name="instruction",
        model_alias=MODEL_ALIAS,
        system_prompt=(
            "You are an expert petroleum engineering educator and technical writer. "
            "You create precise, technically accurate questions and instructions "
            "for training AI models on oil and gas domain knowledge."
        ),
        prompt=(
            "Generate a single, specific petroleum engineering question or instruction "
            "that matches these parameters:\n\n"
            "Category: {{ instruction_category }}\n"
            "Complexity: {{ complexity_level }}\n"
            "Subdomain: {{ subdomain }}\n\n"
            "Guidelines:\n"
            "- The question should be self-contained and specific\n"
            "- For 'calculation_problem': include realistic numerical parameters\n"
            "- For 'troubleshooting': describe a specific operational scenario\n"
            "- For 'comparison_analysis': name specific methods or technologies\n"
            "- For 'design_recommendation': provide realistic well/field conditions\n"
            "- Match the complexity level (beginner=basic concepts, expert=advanced analysis)\n\n"
            "Output ONLY the question/instruction text, nothing else."
        ),
    )
)

# ─── LLM: Generate response ───
config_ir.add_column(
    dd.LLMTextColumnConfig(
        name="response",
        model_alias=MODEL_ALIAS,
        system_prompt=(
            "You are a senior petroleum engineer with 25+ years of field and academic experience. "
            "You provide detailed, technically accurate responses that include specific values, "
            "equations, industry standards (API, SPE, ISO), and practical field considerations. "
            "Use proper engineering units (both field and SI where appropriate)."
        ),
        prompt=(
            "Provide a comprehensive, technically accurate response to the following "
            "petroleum engineering question.\n\n"
            "Question: {{ instruction }}\n\n"
            "Requirements:\n"
            "- Complexity target: {{ complexity_level }}\n"
            "- Include specific technical details, equations, and industry standards\n"
            "- Reference real methods, tools, or software where relevant\n"
            "- Include practical considerations and common field challenges\n"
            "- For calculations, show the full working with realistic values\n"
            "- Aim for 200-500 words depending on complexity\n\n"
            "Provide the response directly without preamble."
        ),
    )
)

print("✅ Pipeline 1 configured: Instruction-Response generation")
print(f"   Model: {MODEL_ALIAS}")
print(f"   Columns: instruction_category, complexity_level, subdomain → instruction → response")

✅ Pipeline 1 configured: Instruction-Response generation
   Model: fast-nvidia-text
   Columns: instruction_category, complexity_level, subdomain → instruction → response


In [ ]:
# ============================================================
# PREVIEW PIPELINE 1 (small test before full run)
# ============================================================

print("🔍 Generating preview (10 sample records)...")
preview_ir = data_designer.preview(config_builder=config_ir)

print("\n" + "="*60)
print("📋 PIPELINE 1 PREVIEW")
print("="*60)
preview_ir.display_sample_record()

df_prev = preview_ir.dataset
print(f"\n📏 Avg instruction length: {df_prev['instruction'].str.len().mean():.0f} chars")
print(f"📏 Avg response length:    {df_prev['response'].str.len().mean():.0f} chars")

[23:22:45] [INFO] 👁️ Preview generation in progress
[23:22:45] [INFO] ✅ Validation passed
[23:22:45] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[23:22:45] [INFO] 🩺 Running health checks for models...
[23:22:45] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'fast-nvidia-text'...


🔍 Generating preview (10 sample records)...


[23:22:47] [INFO]   |-- ✅ Passed!
[23:22:47] [INFO] 🎲 Preparing samplers to generate 10 records across 3 columns
[23:22:47] [INFO] 📝 llm-text model config for column 'instruction'
[23:22:47] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[23:22:47] [INFO]   |-- model alias: 'fast-nvidia-text'
[23:22:47] [INFO]   |-- model provider: 'nvidia'
[23:22:47] [INFO]   |-- inference parameters:
[23:22:47] [INFO]   |  |-- generation_type=chat-completion
[23:22:47] [INFO]   |  |-- max_parallel_requests=32
[23:22:47] [INFO]   |  |-- temperature=0.70
[23:22:47] [INFO]   |  |-- top_p=0.95
[23:22:47] [INFO]   |  |-- max_tokens=1024
[23:22:47] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[23:22:47] [INFO] ⏱️ llm-text column 'instruction' will report progress after each record
[23:22:48] [INFO]   |-- 🌑 llm-text column 'instruction' progress: 1/10 (10%) complete, 1 ok, 0 failed, 0.95 rec/s, eta 9.5s
[23:22:48] [INFO]   |-- 🌑 llm-text column 'instruction' progress: 


📋 PIPELINE 1 PREVIEW


                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                 ┃ Value                                                                               ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ instruction_category │ case_study_analysis                                                                 │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ complexity_level     │ beginner                                                                            │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ subdomain            │ well_integrity_safety                                                               │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ instruction          │ Analyze the following case study: A newly drilled oil well in a shallow onshore     │
│                      │ reservoir experiences a sudden loss of circulation during cementing. Identify the   │
│                      │ potential well integrity risks and recommend one basic preventive measure.          │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ response             │ **Case Study Analysis – Sudden Loss of Circulation During Cementing**               │
│                      │                                                                                     │
│                      │ A loss‑of‑circulation (LOC) event while pumping cement in a shallow onshore well    │
│                      │ (depth ≈ 850 m, true vertical depth) indicates that the circulating pressure        │
│                      │ exceeded the formation’s ability to accept fluid. The primary well‑integrity risks  │
│                      │ are:                                                                                │
│                      │                                                                                     │
│                      │ 1. **Channeling and poor zonal isolation** – cement may bypass the target interval, │
│                      │ leaving a high‑permeability pathway for hydrocarbon migration.                      │
│                      │ 2. **Casing deformation or collapse** – excessive pressure can exceed the collapse  │
│                      │ rating of the surface casing (typically API 5CT, grade J55, collapse pressure ≈ 1.5 │
│                      │ × hydrostatic).                                                                     │
│                      │ 3. **Formation fracture** – if the applied pressure exceeds the fracture pressure   │
│                      │ (P_f), the cement slurry can create a fracture, leading to lost‑circulation         │
│                      │ material (LCM) in the formation and possible reservoir damage.                      │
│                      │                                                                                     │
│                      │ **Technical Basis**                                                                 │
│                      │                                                                                     │
│                      │ - **Hydrostatic pressure (P_h)**                                                    │
│                      │   \[                                                                                │
│                      │   P_h = 0.052 \times \rho_{mw} \times D                                             │
│                      │   \]                                                                                │
│   


📏 Avg instruction length: 450 chars
📏 Avg response length:    1561 chars


In [ ]:
# ============================================================
# FULL GENERATION — Pipeline 1 (batched + auto-save to Drive)
# ============================================================
# This cell is SAFE TO RE-RUN. It resumes from the last checkpoint.

print(f"🚀 Pipeline 1: Generating {NUM_IR_RECORDS:,} instruction-response pairs")
print(f"   Batch size: {BATCH_SIZE:,} | Auto-save to: {P1_CKPT_DIR}")
print(f"   Estimated LLM calls: ~{NUM_IR_RECORDS * 2:,}")
print()

df_ir = generate_batched(
    data_designer=data_designer,
    config_builder=config_ir,
    total_records=NUM_IR_RECORDS,
    batch_size=BATCH_SIZE,
    ckpt_dir=P1_CKPT_DIR,
    dataset_name="petroleum_ir",
)

print(f"\n✅ Pipeline 1 complete: {len(df_ir):,} instruction-response pairs")
print(f"💾 All batches saved to: {P1_CKPT_DIR}")

# Also save the combined file
if len(df_ir) > 0:
    combined_path = AUGMENTED_DIR / "pipeline1_instruction_response.parquet"
    df_ir.to_parquet(combined_path, index=False)
    combined_jsonl = AUGMENTED_DIR / "pipeline1_instruction_response.jsonl"
    with jsonlines.open(combined_jsonl, mode='w') as w:
        for _, row in df_ir.iterrows():
            w.write(row.to_dict())
    print(f"💾 Combined file: {combined_path}")
    print(f"💾 JSONL backup: {combined_jsonl}")

[23:22:56] [INFO] 🎨 Creating Data Designer dataset
[23:22:57] [INFO] 📂 Dataset path '/content/artifacts/petroleum_ir_batch001' already exists. Dataset from this session
		     will be saved to '/content/artifacts/petroleum_ir_batch001_03-17-2026_232257' instead.
[23:22:57] [INFO] ✅ Validation passed
[23:22:57] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[23:22:57] [INFO] 🩺 Running health checks for models...
[23:22:57] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'fast-nvidia-text'...


🚀 Pipeline 1: Generating 20,000 instruction-response pairs
   Batch size: 1,000 | Auto-save to: /content/drive/MyDrive/petroleum_corpus/augmented/checkpoints/pipeline1
   Estimated LLM calls: ~40,000

🚀 Starting fresh: 20,000 records in batches of 1,000

📦 Batch 1 — generating 1,000 records...
   Progress: 0 / 20,000 (0.0%)


[23:22:57] [INFO]   |-- ✅ Passed!
[23:22:57] [INFO] ⏳ Processing batch 1 of 1
[23:22:57] [INFO] 🎲 Preparing samplers to generate 1000 records across 3 columns
[23:22:57] [INFO] 📝 llm-text model config for column 'instruction'
[23:22:57] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[23:22:57] [INFO]   |-- model alias: 'fast-nvidia-text'
[23:22:57] [INFO]   |-- model provider: 'nvidia'
[23:22:57] [INFO]   |-- inference parameters:
[23:22:57] [INFO]   |  |-- generation_type=chat-completion
[23:22:57] [INFO]   |  |-- max_parallel_requests=32
[23:22:57] [INFO]   |  |-- temperature=0.70
[23:22:57] [INFO]   |  |-- top_p=0.95
[23:22:57] [INFO]   |  |-- max_tokens=1024
[23:22:57] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[23:22:57] [INFO] ⏱️ llm-text column 'instruction' will report progress every 100 records
[23:23:09] [INFO]   |-- 🌧️ llm-text column 'instruction' progress: 100/1000 (10%) complete, 100 ok, 0 failed, 8.43 rec/s, eta 106.8s
[23:23:20] 

   ✅ Batch 1: 997 records in 274s (218 rec/min)
   💾 Saved to Drive: batch_001.parquet
   📊 Total progress: 997 / 20,000 (5.0%)
   ⏱️  ETA: ~87 min remaining

📦 Batch 2 — generating 1,000 records...
   Progress: 997 / 20,000 (5.0%)


[23:27:31] [INFO]   |-- ✅ Passed!
[23:27:31] [INFO] ⏳ Processing batch 1 of 1
[23:27:31] [INFO] 🎲 Preparing samplers to generate 1000 records across 3 columns
[23:27:31] [INFO] 📝 llm-text model config for column 'instruction'
[23:27:31] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[23:27:31] [INFO]   |-- model alias: 'fast-nvidia-text'
[23:27:31] [INFO]   |-- model provider: 'nvidia'
[23:27:31] [INFO]   |-- inference parameters:
[23:27:31] [INFO]   |  |-- generation_type=chat-completion
[23:27:31] [INFO]   |  |-- max_parallel_requests=32
[23:27:31] [INFO]   |  |-- temperature=0.70
[23:27:31] [INFO]   |  |-- top_p=0.95
[23:27:31] [INFO]   |  |-- max_tokens=1024
[23:27:31] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[23:27:31] [INFO] ⏱️ llm-text column 'instruction' will report progress every 100 records
[23:27:44] [INFO]   |-- 🌧️ llm-text column 'instruction' progress: 100/1000 (10%) complete, 100 ok, 0 failed, 8.25 rec/s, eta 109.1s
[23:27:54] 

...

   ✅ Batch 19: 996 records in 270s (222 rec/min)
   💾 Saved to Drive: batch_019.parquet
   📊 Total progress: 18,914 / 20,000 (94.6%)
   ⏱️  ETA: ~5 min remaining

📦 Batch 20 — generating 1,000 records...
   Progress: 18,914 / 20,000 (94.6%)


[00:50:18] [INFO]   |-- ✅ Passed!
[00:50:18] [INFO] ⏳ Processing batch 1 of 1
[00:50:18] [INFO] 🎲 Preparing samplers to generate 1000 records across 3 columns
[00:50:18] [INFO] 📝 llm-text model config for column 'instruction'
[00:50:18] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[00:50:18] [INFO]   |-- model alias: 'fast-nvidia-text'
[00:50:18] [INFO]   |-- model provider: 'nvidia'
[00:50:18] [INFO]   |-- inference parameters:
[00:50:18] [INFO]   |  |-- generation_type=chat-completion
[00:50:18] [INFO]   |  |-- max_parallel_requests=32
[00:50:18] [INFO]   |  |-- temperature=0.70
[00:50:18] [INFO]   |  |-- top_p=0.95
[00:50:18] [INFO]   |  |-- max_tokens=1024
[00:50:18] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[00:50:18] [INFO] ⏱️ llm-text column 'instruction' will report progress every 100 records
[00:50:30] [INFO]   |-- 🌧️ llm-text column 'instruction' progress: 100/1000 (10%) complete, 100 ok, 0 failed, 8.14 rec/s, eta 110.6s
[00:50:41] 

   ✅ Batch 20: 997 records in 268s (223 rec/min)
   💾 Saved to Drive: batch_020.parquet
   📊 Total progress: 19,911 / 20,000 (99.6%)
   ⏱️  ETA: ~0 min remaining

📦 Batch 21 — generating 89 records...
   Progress: 19,911 / 20,000 (99.6%)


[00:54:45] [INFO]   |-- ✅ Passed!
[00:54:45] [INFO] ⏳ Processing batch 1 of 1
[00:54:45] [INFO] 🎲 Preparing samplers to generate 89 records across 3 columns
[00:54:45] [INFO] 📝 llm-text model config for column 'instruction'
[00:54:45] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[00:54:45] [INFO]   |-- model alias: 'fast-nvidia-text'
[00:54:45] [INFO]   |-- model provider: 'nvidia'
[00:54:45] [INFO]   |-- inference parameters:
[00:54:45] [INFO]   |  |-- generation_type=chat-completion
[00:54:45] [INFO]   |  |-- max_parallel_requests=32
[00:54:45] [INFO]   |  |-- temperature=0.70
[00:54:45] [INFO]   |  |-- top_p=0.95
[00:54:45] [INFO]   |  |-- max_tokens=1024
[00:54:45] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[00:54:45] [INFO] ⏱️ llm-text column 'instruction' will report progress every 8 records
[00:54:48] [INFO]   |-- 🥚 llm-text column 'instruction' progress: 8/89 (9%) complete, 8 ok, 0 failed, 2.73 rec/s, eta 29.7s
[00:54:49] [INFO]   |-- 

   ✅ Batch 21: 89 records in 36s (148 rec/min)
   💾 Saved to Drive: batch_021.parquet
   📊 Total progress: 20,000 / 20,000 (100.0%)
   ⏱️  ETA: ~0 min remaining

✅ Generation complete! Session time: 92.4 min
   📦 Loaded 20,000 records from 21 checkpoint files.

✅ Pipeline 1 complete: 20,000 instruction-response pairs
💾 All batches saved to: /content/drive/MyDrive/petroleum_corpus/augmented/checkpoints/pipeline1
💾 Combined file: /content/drive/MyDrive/petroleum_corpus/augmented/pipeline1_instruction_response.parquet
💾 JSONL backup: /content/drive/MyDrive/petroleum_corpus/augmented/pipeline1_instruction_response.jsonl


---
## 5. Pipeline 2: Seed-Grounded QA from Scraped Corpus

Uses the actual scraped text as context to generate QA pairs
grounded in real petroleum engineering literature.

In [ ]:
# ============================================================
# CONFIGURE PIPELINE 2
# ============================================================
config_ir = dd.DataDesignerConfigBuilder()
config_seed = dd.DataDesignerConfigBuilder()
# Register custom model if defined
if CUSTOM_MODEL:
    config_ir.add_model_config(CUSTOM_MODEL)
    config_seed.add_model_config(CUSTOM_MODEL)
# ─── Load seed dataset ───
config_seed.with_seed_dataset(
    seed_source={"seed_type": "local", "path": str(seed_path)},
    sampling_strategy="shuffle",
)

# ─── Sampler: QA style ───
config_seed.add_column(
    dd.SamplerColumnConfig(
        name="qa_style",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "summarize_key_points",
                "explain_methodology",
                "extract_technical_details",
                "identify_applications",
                "compare_with_alternatives",
                "troubleshoot_scenario",
                "design_based_on_context",
                "evaluate_limitations",
            ],
        ),
    )
)

# ─── LLM: Generate question from seed text ───
config_seed.add_column(
    dd.LLMTextColumnConfig(
        name="instruction",
        model_alias=MODEL_ALIAS,
        system_prompt=(
            "You are an expert petroleum engineering educator creating training data "
            "for a domain-specific AI model. Generate questions that test deep "
            "understanding of petroleum engineering concepts."
        ),
        prompt=(
            "Based on the following petroleum engineering text, generate a specific, "
            "detailed question that can be fully answered using the information provided.\n\n"
            "Source: {{ source }} | Type: {{ content_type }}\n"
            "Title: {{ title }}\n\n"
            "Text:\n{{ text }}\n\n"
            "Question style: {{ qa_style }}\n\n"
            "Generate ONLY the question. Make it specific and technical."
        ),
    )
)

# ─── LLM: Generate answer from seed text ───
config_seed.add_column(
    dd.LLMTextColumnConfig(
        name="response",
        model_alias=MODEL_ALIAS,
        system_prompt=(
            "You are a senior petroleum engineer providing precise, detailed answers. "
            "Base your answer on the provided context but expand with your domain expertise. "
            "Include specific technical details, equations, and practical considerations."
        ),
        prompt=(
            "Answer the following petroleum engineering question using the provided context. "
            "Expand beyond the context where your domain expertise allows, but stay accurate.\n\n"
            "Context from {{ source }}:\n{{ text }}\n\n"
            "Question: {{ instruction }}\n\n"
            "Provide a comprehensive, technically accurate answer (200-500 words)."
        ),
    )
)

print("✅ Pipeline 2 configured: Seed-grounded QA generation")
print(f"   Seed dataset: {seed_path} ({len(seed_df):,} chunks)")
print(f"   Model: {MODEL_ALIAS}")

✅ Pipeline 2 configured: Seed-grounded QA generation
   Seed dataset: /content/drive/MyDrive/petroleum_corpus/processed/seed_corpus.parquet (66,905 chunks)
   Model: fast-nvidia-text


In [ ]:
# ============================================================
# PREVIEW PIPELINE 2
# ============================================================

print("🔍 Generating preview for seed-grounded QA...")
preview_seed = data_designer.preview(config_builder=config_seed)

print("\n" + "="*60)
print("📋 PIPELINE 2 PREVIEW")
print("="*60)
preview_seed.display_sample_record()

df_prev2 = preview_seed.dataset
print(f"\n📏 Avg instruction length: {df_prev2['instruction'].str.len().mean():.0f} chars")
print(f"📏 Avg response length:    {df_prev2['response'].str.len().mean():.0f} chars")

[11:17:01] [INFO] 📺 Preview generation in progress


🔍 Generating preview for seed-grounded QA...


[11:17:02] [INFO] ✅ Validation passed
[11:17:03] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[11:17:03] [INFO] 🩺 Running health checks for models...
[11:17:03] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'fast-nvidia-text'...
[11:17:05] [INFO]   |-- ✅ Passed!
[11:17:06] [INFO] 🌱 Sampling 10 records from seed dataset
[11:17:06] [INFO]   |-- seed dataset size: 66905 records
[11:17:06] [INFO]   |-- sampling strategy: shuffle
[11:17:06] [INFO] 🎲 Preparing samplers to generate 10 records across 1 columns
[11:17:06] [INFO] (💾 + 💾) Concatenating 2 datasets
[11:17:06] [INFO] 📝 llm-text model config for column 'instruction'
[11:17:06] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[11:17:06] [INFO]   |-- model alias: 'fast-nvidia-text'
[11:17:06] [INFO]   |-- model provider: 'nvidia'
[11:17:06] [INFO]   |-- inference parameters:
[11:17:06] [INFO]   |  |-- generation_type=chat-completion
[11:17:06] [INFO]   |  |-- m


📋 PIPELINE 2 PREVIEW


                                                 Seed Columns                                                 
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name         ┃ Value                                                                                       ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ text         │ the high resolution (or vice versa), as illustrated in Fig.1. • A thermodynamic force,      │
│              │ defined via first principles of thermodynamics, acts on the COM of each molecule and a      │
│              │ thermostat is added to assure the overall thermodynamic equilibrium at the chosen           │
│              │ temperature. The thermodynamic force is derived in such a way that: pAT + ρ0 ∫ ∆ F th(r)dr  │
│              │ = pCG, where pAT is the chosen pressure of the atomistic system (region), pCG is the        │
│              │ pressure of the coarse-grained model, ρ0 is the chosen molecular density of the atomistic   │
│              │ system (region) [30] (the explicit expression of F th(r) will be specified later on). A     │
│              │ thermostat is added to take care of the loss/gain of energy in the transition region. This  │
│              │ is the first step to pass from the original intuitive idea of AdResS to a well founded      │
│              │ Grand Canonical framework of the method. In the original AdResS set-up, the thermostat acts │
│              │ over the whole system (see top panel of Fig.1), in this work the idea has been developed    │
│              │ further and in 11 order to match the requirements of the reservoir of the BL model for the  │
│              │ calculation of equilibrium time correlation functions, we have constructed a set-up, in     │
│              │ which the thermostat is applied to the reservoir only (i.e. hybrid and coarse-grained       │
│              │ region); see bottom panel of Fig.1. In Ref.[24] and in Ref.[21] necessary conditions in ∆   │
│              │ were derived so that the spatial prob- ability distribution in the atomistic region was     │
│              │ close to that of a fully atomistic reference system up to a certain chosen order. The       │
│              │ probability distribution is that of a Grand Canon- ical ensemble, hence the name            │
│              │ Grand-Canonical-AdResS (GC-AdResS). We define the m-th order statistics of a joint          │
│              │ probability distribution of M molecules, p(r 1, · · · , rM), as p(m)(r 1, · · · , rm) = ∫   │
│              │ p(r 1, · · · , rm, rm+1, · · · , rM) drm+1 · · · drN . (10) The molecular number density    │
│              │ ρ(r) corresponds to the first order, the radial distribution function to the second,        │
│              │ three-body distributions to the third order statistics and so on; examples of how the       │
│              │ statistics in the atomistic region is reproduced will be shown later on. We emphasize that, │
│              │ by construction of the method, the accuracy in the atomistic region is independent of the   │
│              │ accuracy of the coarse-grained model, thus, in the coarse-grained region, one can use a     │
│              │ generic liquid of spheres whose only requirement is that it has the same molecular density  │
│              │ of the reference system (i.e. we need only to know the distribution of the reservoir and    │
│              │ not its microscopic details, which is in accordance with the basic principle of             │
│              │ construction of the BL reservoir). It was numerically demonstrated for the case of liquid   │
│              │ water that the target Grand Canonical distribution, numerically defined as the probability  │
│              │ distribution of a subsystem (of the size of the atomistic region in GC-AdResS) in a large,  │
│   


📏 Avg instruction length: 286 chars
📏 Avg response length:    3031 chars


In [ ]:
# ============================================================
# FULL GENERATION — Pipeline 2 (batched + auto-save to Drive)
# ============================================================
# This cell is SAFE TO RE-RUN. It resumes from the last checkpoint.

print(f"🚀 Pipeline 2: Generating {NUM_SEED_RECORDS:,} seed-grounded QA pairs")
print(f"   Batch size: {BATCH_SIZE:,} | Auto-save to: {P2_CKPT_DIR}")
print()

df_seed_qa = generate_batched(
    data_designer=data_designer,
    config_builder=config_seed,
    total_records=NUM_SEED_RECORDS,
    batch_size=BATCH_SIZE,
    ckpt_dir=P2_CKPT_DIR,
    dataset_name="petroleum_seed_qa",
)

print(f"\n✅ Pipeline 2 complete: {len(df_seed_qa):,} seed-grounded QA pairs")
print(f"💾 All batches saved to: {P2_CKPT_DIR}")

# Also save the combined file
if len(df_seed_qa) > 0:
    combined_path = AUGMENTED_DIR / "pipeline2_seed_grounded_qa.parquet"
    df_seed_qa.to_parquet(combined_path, index=False)
    combined_jsonl = AUGMENTED_DIR / "pipeline2_seed_grounded_qa.jsonl"
    with jsonlines.open(combined_jsonl, mode='w') as w:
        for _, row in df_seed_qa.iterrows():
            w.write(row.to_dict())
    print(f"💾 Combined file: {combined_path}")
    print(f"💾 JSONL backup: {combined_jsonl}")

[11:17:38] [INFO] 🎨 Creating Data Designer dataset
[11:17:38] [INFO] ✅ Validation passed
[11:17:38] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[11:17:38] [INFO] 🩺 Running health checks for models...
[11:17:38] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'fast-nvidia-text'...


🚀 Pipeline 2: Generating 20,000 seed-grounded QA pairs
   Batch size: 1,000 | Auto-save to: /content/drive/MyDrive/petroleum_corpus/augmented/checkpoints/pipeline2

🚀 Starting fresh: 20,000 records in batches of 1,000

📦 Batch 1 — generating 1,000 records...
   Progress: 0 / 20,000 (0.0%)


[11:17:39] [INFO]   |-- ✅ Passed!
[11:17:39] [INFO] ⏳ Processing batch 1 of 1
[11:17:40] [INFO] 🌱 Sampling 1000 records from seed dataset
[11:17:40] [INFO]   |-- seed dataset size: 66905 records
[11:17:40] [INFO]   |-- sampling strategy: shuffle
[11:17:40] [INFO] 🎲 Preparing samplers to generate 1000 records across 1 columns
[11:17:40] [INFO] (💾 + 💾) Concatenating 2 datasets
[11:17:40] [INFO] 📝 llm-text model config for column 'instruction'
[11:17:40] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[11:17:40] [INFO]   |-- model alias: 'fast-nvidia-text'
[11:17:40] [INFO]   |-- model provider: 'nvidia'
[11:17:40] [INFO]   |-- inference parameters:
[11:17:40] [INFO]   |  |-- generation_type=chat-completion
[11:17:40] [INFO]   |  |-- max_parallel_requests=32
[11:17:40] [INFO]   |  |-- temperature=0.70
[11:17:40] [INFO]   |  |-- top_p=0.95
[11:17:40] [INFO]   |  |-- max_tokens=1024
[11:17:40] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[11:17:40] [INF

   ✅ Batch 1: 979 records in 282s (208 rec/min)
   💾 Saved to Drive: batch_001.parquet
   📊 Total progress: 979 / 20,000 (4.9%)
   ⏱️  ETA: ~91 min remaining

📦 Batch 2 — generating 1,000 records...
   Progress: 979 / 20,000 (4.9%)


[11:22:21] [INFO]   |-- ✅ Passed!
[11:22:21] [INFO] ⏳ Processing batch 1 of 1
[11:22:23] [INFO] 🌱 Sampling 1000 records from seed dataset
[11:22:23] [INFO]   |-- seed dataset size: 66905 records
[11:22:23] [INFO]   |-- sampling strategy: shuffle
[11:22:23] [INFO] 🎲 Preparing samplers to generate 1000 records across 1 columns
[11:22:23] [INFO] (💾 + 💾) Concatenating 2 datasets
[11:22:23] [INFO] 📝 llm-text model config for column 'instruction'
[11:22:23] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[11:22:23] [INFO]   |-- model alias: 'fast-nvidia-text'
[11:22:23] [INFO]   |-- model provider: 'nvidia'
[11:22:23] [INFO]   |-- inference parameters:
[11:22:23] [INFO]   |  |-- generation_type=chat-completion
[11:22:23] [INFO]   |  |-- max_parallel_requests=32
[11:22:23] [INFO]   |  |-- temperature=0.70
[11:22:23] [INFO]   |  |-- top_p=0.95
[11:22:23] [INFO]   |  |-- max_tokens=1024
[11:22:23] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[11:22:23] [INF

...

   ✅ Batch 20: 982 records in 259s (227 rec/min)
   💾 Saved to Drive: batch_020.parquet
   📊 Total progress: 19,673 / 20,000 (98.4%)
   ⏱️  ETA: ~2 min remaining

📦 Batch 21 — generating 327 records...
   Progress: 19,673 / 20,000 (98.4%)


[12:48:44] [INFO]   |-- ✅ Passed!
[12:48:44] [INFO] ⏳ Processing batch 1 of 1
[12:48:46] [INFO] 🌱 Sampling 327 records from seed dataset
[12:48:46] [INFO]   |-- seed dataset size: 66905 records
[12:48:46] [INFO]   |-- sampling strategy: shuffle
[12:48:46] [INFO] 🎲 Preparing samplers to generate 327 records across 1 columns
[12:48:46] [INFO] (💾 + 💾) Concatenating 2 datasets
[12:48:46] [INFO] 📝 llm-text model config for column 'instruction'
[12:48:46] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[12:48:46] [INFO]   |-- model alias: 'fast-nvidia-text'
[12:48:46] [INFO]   |-- model provider: 'nvidia'
[12:48:46] [INFO]   |-- inference parameters:
[12:48:46] [INFO]   |  |-- generation_type=chat-completion
[12:48:46] [INFO]   |  |-- max_parallel_requests=32
[12:48:46] [INFO]   |  |-- temperature=0.70
[12:48:46] [INFO]   |  |-- top_p=0.95
[12:48:46] [INFO]   |  |-- max_tokens=1024
[12:48:46] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[12:48:46] [INFO]

   ✅ Batch 21: 320 records in 98s (195 rec/min)
   💾 Saved to Drive: batch_021.parquet
   📊 Total progress: 19,993 / 20,000 (100.0%)
   ⏱️  ETA: ~0 min remaining

📦 Batch 22 — generating 7 records...
   Progress: 19,993 / 20,000 (100.0%)


[12:50:23] [INFO]   |-- ✅ Passed!
[12:50:23] [INFO] ⏳ Processing batch 1 of 1
[12:50:24] [INFO] 🌱 Sampling 7 records from seed dataset
[12:50:24] [INFO]   |-- seed dataset size: 66905 records
[12:50:24] [INFO]   |-- sampling strategy: shuffle
[12:50:24] [INFO] 🎲 Preparing samplers to generate 7 records across 1 columns
[12:50:24] [INFO] (💾 + 💾) Concatenating 2 datasets
[12:50:24] [INFO] 📝 llm-text model config for column 'instruction'
[12:50:24] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[12:50:24] [INFO]   |-- model alias: 'fast-nvidia-text'
[12:50:24] [INFO]   |-- model provider: 'nvidia'
[12:50:24] [INFO]   |-- inference parameters:
[12:50:24] [INFO]   |  |-- generation_type=chat-completion
[12:50:24] [INFO]   |  |-- max_parallel_requests=32
[12:50:24] [INFO]   |  |-- temperature=0.70
[12:50:24] [INFO]   |  |-- top_p=0.95
[12:50:24] [INFO]   |  |-- max_tokens=1024
[12:50:24] [INFO] ⚡️ Processing llm-text column 'instruction' with 32 concurrent workers
[12:50:24] [INFO] ⏱️ 

   ✅ Batch 22: 7 records in 10s (40 rec/min)
   💾 Saved to Drive: batch_022.parquet
   📊 Total progress: 20,000 / 20,000 (100.0%)
   ⏱️  ETA: ~0 min remaining

✅ Generation complete! Session time: 92.9 min
   📦 Loaded 20,000 records from 22 checkpoint files.

✅ Pipeline 2 complete: 20,000 seed-grounded QA pairs
💾 All batches saved to: /content/drive/MyDrive/petroleum_corpus/augmented/checkpoints/pipeline2
💾 Combined file: /content/drive/MyDrive/petroleum_corpus/augmented/pipeline2_seed_grounded_qa.parquet
💾 JSONL backup: /content/drive/MyDrive/petroleum_corpus/augmented/pipeline2_seed_grounded_qa.jsonl


---
## 6. Pipeline 3: Quality Scoring with LLM-as-Judge

In [ ]:
# ============================================================
# MERGE & PREPARE FOR QUALITY SCORING
# ============================================================

all_generated = []

# Pipeline 1
if 'df_ir' in dir() and df_ir is not None and len(df_ir) > 0:
    df_ir_clean = df_ir[['instruction', 'response']].copy()
    if 'instruction_category' in df_ir.columns:
        df_ir_clean['instruction_category'] = df_ir['instruction_category']
    if 'complexity_level' in df_ir.columns:
        df_ir_clean['complexity_level'] = df_ir['complexity_level']
    if 'subdomain' in df_ir.columns:
        df_ir_clean['subdomain'] = df_ir['subdomain']
    df_ir_clean['pipeline'] = 'knowledge_generation'
    df_ir_clean['source_type'] = 'synthetic'
    all_generated.append(df_ir_clean)
    print(f"✅ Pipeline 1: {len(df_ir_clean):,} records")

# Pipeline 2
if 'df_seed_qa' in dir() and df_seed_qa is not None and len(df_seed_qa) > 0:
    df_seed_clean = df_seed_qa[['instruction', 'response']].copy()
    if 'qa_style' in df_seed_qa.columns:
        df_seed_clean['instruction_category'] = df_seed_qa['qa_style']
    if 'content_type' in df_seed_qa.columns:
        df_seed_clean['source_type'] = df_seed_qa['content_type']
    else:
        df_seed_clean['source_type'] = 'seed_grounded'
    df_seed_clean['pipeline'] = 'seed_grounded_qa'
    all_generated.append(df_seed_clean)
    print(f"✅ Pipeline 2: {len(df_seed_clean):,} records")

if all_generated:
    df_all = pd.concat(all_generated, ignore_index=True)
    print(f"\n📊 Total combined records: {len(df_all):,}")
else:
    print("⚠️  No generated data found. Run pipelines 1 and/or 2 first.")
    df_all = pd.DataFrame()

✅ Pipeline 2: 20,000 records

📊 Total combined records: 20,000


In [ ]:
# ============================================================
# CONFIGURE & RUN QUALITY SCORING
# ============================================================

if len(df_all) > 0:
    JUDGE_SAMPLE_SIZE = min(JUDGE_SAMPLE, len(df_all))
    df_sample = df_all.sample(n=JUDGE_SAMPLE_SIZE, random_state=42).reset_index(drop=True)

    # Save as seed for judge
    judge_seed_path = PROCESSED_DIR / "judge_sample.parquet"
    df_sample.to_parquet(judge_seed_path, index=False)

    config_judge = dd.DataDesignerConfigBuilder()

    # Register custom model if defined
    if CUSTOM_MODEL:
        config_judge.add_model_config(CUSTOM_MODEL)

    config_judge.with_seed_dataset(
        seed_source={"seed_type": "local", "path": str(judge_seed_path)},
        sampling_strategy="ordered",
    )

    config_judge.add_column(
        dd.LLMJudgeColumnConfig(
            name="quality_score",
            model_alias=MODEL_ALIAS,
            prompt=(
                "Evaluate this petroleum engineering instruction-response pair:\n\n"
                "INSTRUCTION: {{ instruction }}\n\n"
                "RESPONSE: {{ response }}"
            ),
            scores=[
                dd.Score(
                    name="technical_accuracy",
                    description="Is the response technically accurate for petroleum engineering?",
                    options={
                        "4": "Completely accurate with proper terminology and values",
                        "3": "Mostly accurate with minor issues",
                        "2": "Partially accurate with some errors",
                        "1": "Mostly inaccurate",
                        "0": "Completely inaccurate or irrelevant"
                    }
                ),
                dd.Score(
                    name="completeness",
                    description="Does the response fully address the instruction?",
                    options={
                        "4": "Comprehensive with all key aspects covered",
                        "3": "Good coverage of most aspects",
                        "2": "Partially addresses the question",
                        "1": "Mostly incomplete",
                        "0": "Does not address the question"
                    }
                ),
                dd.Score(
                    name="usefulness",
                    description="Would this be useful for training a petroleum engineering AI?",
                    options={
                        "4": "Excellent training example with specific details",
                        "3": "Good training example",
                        "2": "Acceptable but could be better",
                        "1": "Low quality for training",
                        "0": "Not useful for training"
                    }
                ),
            ],
        )
    )

    print(f"⚖️ Scoring {JUDGE_SAMPLE_SIZE} records for quality...")

    try:
        judge_results = data_designer.create(
            config_builder=config_judge,
            num_records=JUDGE_SAMPLE_SIZE,
            dataset_name="petroleum_quality_scores",
        )

        df_scored = judge_results.load_dataset()

        # Extract numeric scores
        score_cols = []
        for col in ['quality_score_technical_accuracy', 'quality_score_completeness', 'quality_score_usefulness']:
            if col in df_scored.columns:
                df_scored[col] = pd.to_numeric(df_scored[col], errors='coerce')
                score_cols.append(col)

        if score_cols:
            df_scored['composite_score'] = df_scored[score_cols].mean(axis=1)

            print(f"\n{'='*60}")
            print("📊 QUALITY SCORE DISTRIBUTION")
            print(f"{'='*60}")
            for col in score_cols:
                print(f"   {col:45s}  mean={df_scored[col].mean():.2f}  std={df_scored[col].std():.2f}")
            print(f"   {'composite_score':45s}  mean={df_scored['composite_score'].mean():.2f}")

            high_quality = df_scored[df_scored['composite_score'] >= 3.0]
            print(f"\n   High quality (>=3.0): {len(high_quality):,} / {len(df_scored):,} "
                  f"({100*len(high_quality)/len(df_scored):.1f}%)")

        # Save scored sample to Drive
        scored_path = AUGMENTED_DIR / "quality_scored_sample.parquet"
        df_scored.to_parquet(scored_path, index=False)
        print(f"\n💾 Scored sample saved to: {scored_path}")

    except Exception as e:
        print(f"❌ Quality scoring failed: {e}")
        print("   Skipping scoring — your generated data is still saved.")
        df_scored = pd.DataFrame()
else:
    print("⚠️  No data to score.")

[13:05:16] [INFO] 🎨 Creating Data Designer dataset
[13:05:16] [INFO] ✅ Validation passed
[13:05:16] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[13:05:16] [INFO] 🩺 Running health checks for models...
[13:05:16] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'fast-nvidia-text'...


⚖️ Scoring 500 records for quality...


[13:05:19] [INFO]   |-- ✅ Passed!
[13:05:19] [INFO] ⏳ Processing batch 1 of 1
[13:05:19] [INFO] 🌱 Sampling 500 records from seed dataset
[13:05:19] [INFO]   |-- seed dataset size: 500 records
[13:05:19] [INFO]   |-- sampling strategy: ordered
[13:05:19] [INFO] ⚖️ llm-judge model config for column 'quality_score'
[13:05:19] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[13:05:19] [INFO]   |-- model alias: 'fast-nvidia-text'
[13:05:19] [INFO]   |-- model provider: 'nvidia'
[13:05:19] [INFO]   |-- inference parameters:
[13:05:19] [INFO]   |  |-- generation_type=chat-completion
[13:05:19] [INFO]   |  |-- max_parallel_requests=32
[13:05:19] [INFO]   |  |-- temperature=0.70
[13:05:19] [INFO]   |  |-- top_p=0.95
[13:05:19] [INFO]   |  |-- max_tokens=1024
[13:05:19] [INFO] ⚡️ Processing llm-judge column 'quality_score' with 32 concurrent workers
[13:05:19] [INFO] ⏱️ llm-judge column 'quality_score' will report progress every 50 records
[13:05:29] [INFO]   |-- 🌧️ llm-judge column 'qualit


💾 Scored sample saved to: /content/drive/MyDrive/petroleum_corpus/augmented/quality_scored_sample.parquet


---
## 7. Final Export — Merge, Filter, Format for Fine-Tuning

In [ ]:
# ============================================================
# FINAL EXPORT
# ============================================================

print("📂 Loading all generated data from Drive...")
all_dfs = []

# Load from checkpoint directories (most reliable source)
for ckpt_dir, label in [(P1_CKPT_DIR, "Pipeline 1"), (P2_CKPT_DIR, "Pipeline 2")]:
    ckpt_files = sorted(ckpt_dir.glob("batch_*.parquet"))
    for f in ckpt_files:
        try:
            df = pd.read_parquet(f)
            all_dfs.append(df)
        except Exception:
            pass
    total = sum(len(pd.read_parquet(f)) for f in ckpt_files if f.exists())
    print(f"   ✅ {label}: {total:,} records from {len(ckpt_files)} batches")

if not all_dfs:
    print("⚠️ No generated data found. Run the generation pipelines first.")
else:
    df_final = pd.concat(all_dfs, ignore_index=True)
    print(f"\n📊 Total records before filtering: {len(df_final):,}")

    # ===== Quality Filtering =====
    df_final = df_final.dropna(subset=['instruction', 'response'])
    df_final = df_final[
        (df_final['instruction'].str.len() > 20) &
        (df_final['response'].str.len() > 50)
    ].copy()

    # Deduplicate by instruction
    df_final = df_final.drop_duplicates(subset=['instruction'], keep='first')
    print(f"   After quality filtering: {len(df_final):,}")

    # ===== Format 1: Alpaca =====
    alpaca_path = PROCESSED_DIR / "final_alpaca_format.jsonl"
    with jsonlines.open(alpaca_path, mode='w') as w:
        for _, row in df_final.iterrows():
            w.write({
                "instruction": row['instruction'],
                "input": "",
                "output": row['response'],
            })
    print(f"\n💾 Alpaca format:    {alpaca_path}")

    # ===== Format 2: ShareGPT =====
    sharegpt_path = PROCESSED_DIR / "final_sharegpt_format.jsonl"
    with jsonlines.open(sharegpt_path, mode='w') as w:
        for _, row in df_final.iterrows():
            w.write({
                "conversations": [
                    {"from": "human", "value": row['instruction']},
                    {"from": "gpt", "value": row['response']},
                ]
            })
    print(f"💾 ShareGPT format:  {sharegpt_path}")

    # ===== Format 3: Full Parquet =====
    full_path = PROCESSED_DIR / "final_full_dataset.parquet"
    df_final.to_parquet(full_path, index=False)
    print(f"💾 Full Parquet:     {full_path}")

    # ===== Statistics =====
    word_counts_i = df_final['instruction'].str.split().str.len()
    word_counts_r = df_final['response'].str.split().str.len()

    final_stats = {
        "total_instruction_response_pairs": len(df_final),
        "total_instruction_words": int(word_counts_i.sum()),
        "total_response_words": int(word_counts_r.sum()),
        "avg_instruction_words": int(word_counts_i.mean()),
        "avg_response_words": int(word_counts_r.mean()),
        "generated_at": datetime.now().isoformat(),
        "generated_with": "NVIDIA NeMo Data Designer v2 (batched)",
    }
    if 'pipeline' in df_final.columns:
        final_stats['pipeline_breakdown'] = dict(df_final['pipeline'].value_counts())

    with open(METADATA_DIR / "final_statistics.json", 'w') as f:
        json.dump(final_stats, f, indent=2, default=str)

    print(f"\n{'='*70}")
    print("🏆 FINAL DATASET STATISTICS")
    print(f"{'='*70}")
    print(f"📊 Total pairs:           {final_stats['total_instruction_response_pairs']:,}")
    print(f"📝 Total instruction words: {final_stats['total_instruction_words']:,}")
    print(f"📝 Total response words:    {final_stats['total_response_words']:,}")
    print(f"📐 Avg instruction words:   {final_stats['avg_instruction_words']}")
    print(f"📐 Avg response words:      {final_stats['avg_response_words']}")

    total_drive_size = sum(f.stat().st_size for f in OUTPUT_DIR.rglob('*') if f.is_file())
    print(f"\n💾 Total size on Drive: {total_drive_size/(1024*1024):.1f} MB")
    print(f"📂 Location: Google Drive > My Drive > petroleum_corpus")

📂 Loading all generated data from Drive...
   ✅ Pipeline 1: 20,000 records from 21 batches
   ✅ Pipeline 2: 20,000 records from 22 batches

📊 Total records before filtering: 40,000
   After quality filtering: 33,859

💾 Alpaca format:    /content/drive/MyDrive/petroleum_corpus/processed/final_alpaca_format.jsonl
💾 ShareGPT format:  /content/drive/MyDrive/petroleum_corpus/processed/final_sharegpt_format.jsonl
💾 Full Parquet:     /content/drive/MyDrive/petroleum_corpus/processed/final_full_dataset.parquet

🏆 FINAL DATASET STATISTICS
📊 Total pairs:           33,859
📝 Total instruction words: 1,569,908
📝 Total response words:    12,347,947
📐 Avg instruction words:   46
📐 Avg response words:      364

💾 Total size on Drive: 1851.7 MB
📂 Location: Google Drive > My Drive > petroleum_corpus


In [ ]:
# ============================================================
# FINAL OUTPUT STRUCTURE
# ============================================================

print("\n📁 Output Directory:")
print("=" * 60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    for f in sorted(files):
        fp = Path(root) / f
        size = fp.stat().st_size / (1024*1024)
        print(f"{indent}  📄 {f} ({size:.2f} MB)")

print(f"""
{'='*70}
✅ DATASET GENERATION COMPLETE — NVIDIA Data Designer v2
   ALL FILES SAVED TO GOOGLE DRIVE (safe from Colab disconnects)
{'='*70}

📂 Google Drive location:
   My Drive > petroleum_corpus

📋 Output files:

  FOR FINE-TUNING:
  ├── processed/final_alpaca_format.jsonl    (Alpaca format)
  ├── processed/final_sharegpt_format.jsonl  (ShareGPT format)
  └── processed/final_full_dataset.parquet   (Full metadata)

  CHECKPOINTS (individual batches — your safety net):
  ├── augmented/checkpoints/pipeline1/batch_*.parquet
  └── augmented/checkpoints/pipeline2/batch_*.parquet

  COMBINED (merged from checkpoints):
  ├── augmented/pipeline1_instruction_response.parquet
  ├── augmented/pipeline1_instruction_response.jsonl
  ├── augmented/pipeline2_seed_grounded_qa.parquet
  └── augmented/pipeline2_seed_grounded_qa.jsonl

💡 KEY FEATURES OF THIS NOTEBOOK:
  • Every batch is auto-saved to Drive — no data loss on disconnect
  • Re-run any generation cell to resume from last checkpoint
  • Higher parallelism for faster generation (buffer_size=2000)
  • Error recovery: retries failed batches automatically

📋 Next steps:
  1. Load final_alpaca_format.jsonl or final_sharegpt_format.jsonl
  2. Fine-tune with Unsloth + Llama 3.1 / Mistral / Qwen
  3. Use quality scores to further filter if needed
""")


📁 Output Directory:
📂 petroleum_corpus/
  📂 raw/
    📄 01_arxiv.jsonl (2.42 MB)
    📄 02_semantic_scholar.jsonl (38.51 MB)
    📄 03_openalex.jsonl (107.08 MB)
    📄 04_crossref.jsonl (35.26 MB)
    📄 05_doe_osti.jsonl (17.81 MB)
    📄 06_core.jsonl (16.36 MB)
    📄 07_doaj.jsonl (13.00 MB)
    📄 08_unpaywall.jsonl (2.26 MB)
    📄 09_petrowiki.jsonl (0.00 MB)
    📄 10_slb_glossary.jsonl (0.92 MB)
    📄 11_wikipedia.jsonl (1.43 MB)
    📄 12_eia.jsonl (47.41 MB)
    📄 13_epa.jsonl (0.05 MB)
    📄 13_jpt.jsonl (0.18 MB)
    📄 13_rigzone.jsonl (0.05 MB)
    📄 13_usgs.jsonl (0.23 MB)
    📄 14_onepetro.jsonl (13.18 MB)
    📄 15_datasets.jsonl (0.01 MB)
  📂 processed/
    📄 consolidated_corpus.jsonl (290.47 MB)
    📄 final_alpaca_format.jsonl (102.00 MB)
    📄 final_full_dataset.parquet (86.00 MB)
    📄 final_sharegpt_format.jsonl (103.13 MB)
    📄 judge_sample.parquet (1.01 MB)
    📄 seed_corpus.csv (169.98 MB)
    📄 seed_corpus.jsonl (191.71 MB)
    📄 seed_corpus.parquet (91.21 MB)
  📂 meta

---
## 🔧 Utility: Reset Checkpoints (if needed)

Run the cell below **only** if you want to start generation from scratch
for a specific pipeline. This deletes the checkpoint files.

In [ ]:
# ============================================================
# RESET CHECKPOINTS (uncomment the pipeline you want to reset)
# ============================================================
# WARNING: This deletes checkpoint files. Only run if you want to regenerate.

# import shutil

# # Reset Pipeline 1 checkpoints
# shutil.rmtree(P1_CKPT_DIR, ignore_errors=True)
# P1_CKPT_DIR.mkdir(parents=True, exist_ok=True)
# print("🗑️  Pipeline 1 checkpoints cleared.")

# # Reset Pipeline 2 checkpoints
# shutil.rmtree(P2_CKPT_DIR, ignore_errors=True)
# P2_CKPT_DIR.mkdir(parents=True, exist_ok=True)
# print("🗑️  Pipeline 2 checkpoints cleared.")